# Financial ML Models — 4 Core Algorithms
### 1 Use Case Each · Theory + Math + Implementation + Evaluation

| # | Model | Type | Financial Use Case |
|---|---|---|---|
| 1 | Linear Regression | Regression | Loan Amount Prediction |
| 2 | Logistic Regression | Binary Classification | Loan Default Prediction |
| 3 | Decision Tree | Multi-class Classification | Investment Risk Category |
| 4 | Support Vector Machine (SVM) | Non-linear Classification | Credit Card Fraud Detection |

Each section covers: **Theory → Mathematics → Financial Dataset → Implementation → Evaluation → Interpretation**.


## 0. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, roc_auc_score, ConfusionMatrixDisplay,
    classification_report
)

np.random.seed(42)
plt.rcParams["figure.dpi"] = 100
print("Libraries loaded successfully.")

## 1. Linear Regression — Loan Amount Prediction

### 1.1 Theory
- Linear Regression assumes a **straight-line relationship** between input features and a continuous output.
- Here we predict the **loan amount** a bank should approve based on a customer's financial profile.
- The model finds the best-fit line by minimizing total squared error (**Ordinary Least Squares**).


### 1.2 Mathematics

$$\hat{y} = \beta_0 + \beta_1x_1 + \beta_2x_2 + ... + \beta_nx_n + \varepsilon$$

**Cost function (Mean Squared Error):**
$$MSE = \frac{1}{n}\sum(y_i - \hat{y}_i)^2$$

Training finds the $\beta$ coefficients that minimize MSE.


### 1.3 Financial Dataset

In [ ]:
n = 1000
income = np.random.normal(60000, 20000, n).clip(15000, 200000)
credit_score = np.random.normal(650, 80, n).clip(300, 850)
existing_debt = np.random.normal(15000, 8000, n).clip(0, 80000)
years_employed = np.random.normal(6, 4, n).clip(0, 30)

loan_amount = (
    0.35 * income
    + 120 * credit_score
    - 0.5 * existing_debt
    + 800 * years_employed
    + np.random.normal(0, 8000, n)
)
loan_amount = np.clip(loan_amount, 0, None)

df_lr = pd.DataFrame({
    "Income": income, "CreditScore": credit_score,
    "ExistingDebt": existing_debt, "YearsEmployed": years_employed,
    "LoanAmount": loan_amount
})
df_lr.head()

### 1.4 Implementation

In [ ]:
X = df_lr[["Income", "CreditScore", "ExistingDebt", "YearsEmployed"]]
y = df_lr["LoanAmount"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lin_model = LinearRegression()
lin_model.fit(X_train, y_train)
y_pred_lr = lin_model.predict(X_test)

r2 = r2_score(y_test, y_pred_lr)
mae = mean_absolute_error(y_test, y_pred_lr)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))

print("--- Linear Regression: Loan Amount Prediction ---")
print(f"R2 Score : {r2:.4f}")
print(f"MAE      : {mae:.2f}")
print(f"RMSE     : {rmse:.2f}")

### 1.5 Evaluation & Interpretation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test, y_pred_lr, alpha=0.4, color="#2a78d6")
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", linewidth=2)
axes[0].set_xlabel("Actual Loan Amount")
axes[0].set_ylabel("Predicted Loan Amount")
axes[0].set_title(f"Actual vs Predicted (R\u00b2 = {r2:.3f})")
axes[0].grid(alpha=0.3)

coefs = pd.Series(lin_model.coef_, index=X.columns).sort_values()
axes[1].barh(coefs.index, coefs.values, color="#1C7293")
axes[1].set_title("Feature Coefficients")
axes[1].set_xlabel("Effect on Loan Amount")
axes[1].grid(alpha=0.3, axis="x")

plt.tight_layout()
plt.show()

**Interpretation:** Income and Credit Score have the strongest positive effect on the predicted
loan amount, while Existing Debt pulls it down — matching real-world underwriting logic.

## 2. Logistic Regression — Loan Default Prediction

### 2.1 Theory
- Predicts a **binary outcome**: will a customer default on their loan (1) or repay it (0)?
- Instead of predicting the outcome directly, it models the **log-odds** as a linear function of
  inputs, then squashes that score into a probability using the **sigmoid function**.


### 2.2 Mathematics

$$p = \frac{1}{1+e^{-z}} \quad \text{where } z = \beta_0 + \beta_1x_1 + ... + \beta_nx_n$$

**Cost function (Log Loss):**
$$L = -\frac{1}{n}\sum[y_i \log(p_i) + (1-y_i)\log(1-p_i)]$$

A threshold (commonly 0.5) converts probability $p$ into a class label.


### 2.3 Financial Dataset

In [ ]:
n = 1200
debt_to_income = np.random.uniform(0, 1, n)
credit_score = np.random.normal(620, 90, n).clip(300, 850)
missed_payments = np.random.poisson(1.2, n)
loan_to_value = np.random.uniform(0.3, 1.2, n)

default_score = (
    2.0 * debt_to_income
    - 0.015 * (credit_score - 600)
    + 0.5 * missed_payments
    + 1.0 * loan_to_value
    - 3.8
)
default_prob = 1 / (1 + np.exp(-default_score))
defaulted = (np.random.rand(n) < default_prob).astype(int)

df_log = pd.DataFrame({
    "DebtToIncome": debt_to_income, "CreditScore": credit_score,
    "MissedPayments": missed_payments, "LoanToValue": loan_to_value,
    "Defaulted": defaulted
})
print("Default rate in dataset:", round(df_log['Defaulted'].mean(), 4))
df_log.head()

### 2.4 Implementation

In [ ]:
X = df_log[["DebtToIncome", "CreditScore", "MissedPayments", "LoanToValue"]]
y = df_log["Defaulted"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

log_model = LogisticRegression()
log_model.fit(X_train_s, y_train)
y_pred_log = log_model.predict(X_test_s)
y_proba_log = log_model.predict_proba(X_test_s)[:, 1]

print("--- Logistic Regression: Loan Default Prediction ---")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_log):.4f}")
print(f"Precision : {precision_score(y_test, y_pred_log):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_log):.4f}")
print(f"F1 Score  : {f1_score(y_test, y_pred_log):.4f}")
print(f"ROC-AUC   : {roc_auc_score(y_test, y_proba_log):.4f}")

### 2.5 Evaluation & Interpretation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred_log)
ConfusionMatrixDisplay(cm, display_labels=["Repaid", "Default"]).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Confusion Matrix")

fpr, tpr, _ = roc_curve(y_test, y_proba_log)
axes[1].plot(fpr, tpr, color="#2a78d6", linewidth=2, label=f"AUC = {roc_auc_score(y_test, y_proba_log):.3f}")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Interpretation:** High Debt-to-Income ratio and Missed Payments push the predicted probability
of default upward, while a higher Credit Score pulls it down — the model's coefficients align with
standard credit-risk intuition.

## 3. Decision Tree — Investment Risk Category Classification

### 3.1 Theory
- Classifies an investment portfolio into a **risk category**: Low, Medium, or High.
- The tree asks a sequence of yes/no questions on features, splitting the data at each step to
  create the **purest possible groups**.
- Purity is measured using **Gini Impurity** or **Entropy**.


### 3.2 Mathematics

$$Gini = 1 - \sum(p_i)^2 \qquad Entropy = -\sum p_i \log_2(p_i)$$

**Information Gain** (used to choose the best split):
$$IG = H(parent) - \sum \frac{n_k}{n}H(child_k)$$

The split with the highest Information Gain is chosen at each node.


### 3.3 Financial Dataset

In [ ]:
n = 1200
volatility = np.random.uniform(0.05, 0.6, n)          # annualized volatility
equity_allocation = np.random.uniform(0, 1, n)         # % of portfolio in equities
leverage = np.random.uniform(1, 4, n)                  # leverage ratio
diversification = np.random.uniform(0, 1, n)           # 0 = concentrated, 1 = diversified

risk_score = (
    2.5 * volatility
    + 1.2 * equity_allocation
    + 0.6 * leverage
    - 1.5 * diversification
    + np.random.normal(0, 0.15, n)
)

risk_category = pd.cut(
    risk_score,
    bins=[-np.inf, np.percentile(risk_score, 33), np.percentile(risk_score, 66), np.inf],
    labels=["Low", "Medium", "High"]
)

df_tree = pd.DataFrame({
    "Volatility": volatility, "EquityAllocation": equity_allocation,
    "Leverage": leverage, "Diversification": diversification,
    "RiskCategory": risk_category
})
print(df_tree["RiskCategory"].value_counts())
df_tree.head()

### 3.4 Implementation

In [ ]:
X = df_tree[["Volatility", "EquityAllocation", "Leverage", "Diversification"]]
y = df_tree["RiskCategory"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_model.fit(X_train, y_train)
y_pred_tree = tree_model.predict(X_test)

print("--- Decision Tree: Investment Risk Classification ---")
print(f"Accuracy : {accuracy_score(y_test, y_pred_tree):.4f}\n")
print(classification_report(y_test, y_pred_tree))

### 3.5 Evaluation & Interpretation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5), gridspec_kw={"width_ratios": [1, 1.3, 1]})

cm = confusion_matrix(y_test, y_pred_tree, labels=["Low", "Medium", "High"])
ConfusionMatrixDisplay(cm, display_labels=["Low", "Medium", "High"]).plot(ax=axes[0], cmap="Greens", colorbar=False)
axes[0].set_title("Confusion Matrix")

plot_tree(tree_model, max_depth=2, feature_names=X.columns, class_names=tree_model.classes_,
          filled=True, fontsize=7, ax=axes[1])
axes[1].set_title("Decision Tree (top 2 levels)")

importances = pd.Series(tree_model.feature_importances_, index=X.columns).sort_values()
axes[2].barh(importances.index, importances.values, color="#21295C")
axes[2].set_title("Feature Importance")
axes[2].grid(alpha=0.3, axis="x")

plt.tight_layout()
plt.show()

**Interpretation:** Volatility and Equity Allocation are the strongest drivers of risk category —
the tree tends to split on Volatility first, since it separates Low/Medium/High risk portfolios
most cleanly.

## 4. Support Vector Machine — Credit Card Fraud Detection

### 4.1 Theory
- SVM finds the hyperplane that separates classes with the **maximum margin**.
- Fraudulent transactions don't sit in one simple region — they surround the "normal" cluster
  from multiple sides, so a **straight line cannot separate** them well.
- The **RBF (kernel trick)** lets SVM draw a **curved, non-linear boundary** by implicitly mapping
  the data into a higher-dimensional space where it becomes separable.


### 4.2 Mathematics

$$\text{minimize } \frac{1}{2}||w||^2 \quad \text{subject to } y_i(w \cdot x_i + b) \geq 1$$

**RBF Kernel** (measures similarity between two points):
$$K(x_i, x_j) = e^{-\gamma ||x_i - x_j||^2}$$

**Hinge Loss:**
$$L = max(0, 1 - y_i(w \cdot x_i + b))$$

Instead of computing coordinates in the higher dimension directly, the kernel function
computes the *similarity* between points as if they were already mapped there — this is the
"trick" that keeps SVM efficient.


### 4.3 Financial Dataset

Using only 2 features (Transaction Amount, Distance from Home) so the non-linear decision boundary can be visualized directly.

In [ ]:
n_legit, n_fraud = 700, 150
center_amount, center_distance = 120, 20

# "Normal" transactions form a tight cluster near the center (small radius)
angle_legit = np.random.uniform(0, 2 * np.pi, n_legit)
radius_legit = np.random.uniform(0, 3, n_legit)
normal_amount = center_amount + radius_legit * 15 * np.cos(angle_legit)
normal_distance = center_distance + radius_legit * 3 * np.sin(angle_legit)
normal_label = np.zeros(n_legit)

# Fraud transactions form a RING that surrounds the legit cluster from every direction
# -> this cannot be separated by a single straight line (not linearly separable)
angle_fraud = np.random.uniform(0, 2 * np.pi, n_fraud)
radius_fraud = np.random.uniform(5.5, 9, n_fraud)
fraud_amount = center_amount + radius_fraud * 15 * np.cos(angle_fraud)
fraud_distance = center_distance + radius_fraud * 3 * np.sin(angle_fraud)
fraud_label = np.ones(n_fraud)

amount = np.concatenate([normal_amount, fraud_amount]).clip(1, None)
distance = np.concatenate([normal_distance, fraud_distance]).clip(0, None)
is_fraud = np.concatenate([normal_label, fraud_label]).astype(int)

df_svm = pd.DataFrame({"Amount": amount, "Distance": distance, "IsFraud": is_fraud})
print("Fraud rate in dataset:", round(df_svm['IsFraud'].mean(), 4))
df_svm.head()

### 4.4 Implementation — Linear vs RBF Kernel

In [ ]:
X = df_svm[["Amount", "Distance"]]
y = df_svm["IsFraud"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

svm_linear = SVC(kernel="linear", class_weight="balanced")
svm_linear.fit(X_train_s, y_train)
pred_linear = svm_linear.predict(X_test_s)

svm_rbf = SVC(kernel="rbf", gamma="scale", class_weight="balanced")
svm_rbf.fit(X_train_s, y_train)
pred_rbf = svm_rbf.predict(X_test_s)

print("--- SVM: Credit Card Fraud Detection ---")
print(f"Linear Kernel -> Accuracy: {accuracy_score(y_test, pred_linear):.4f}, F1: {f1_score(y_test, pred_linear):.4f}")
print(f"RBF Kernel    -> Accuracy: {accuracy_score(y_test, pred_rbf):.4f}, F1: {f1_score(y_test, pred_rbf):.4f}")

### 4.5 Evaluation & Interpretation — Decision Boundaries

In [ ]:
def plot_boundary(ax, model, X_s, y_true, title):
    x_min, x_max = X_s[:, 0].min() - 0.5, X_s[:, 0].max() + 0.5
    y_min, y_max = X_s[:, 1].min() - 0.5, X_s[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.25, cmap="coolwarm")
    ax.scatter(X_s[y_true == 0, 0], X_s[y_true == 0, 1], s=12, c="#1C7293", label="Legit", alpha=0.6)
    ax.scatter(X_s[y_true == 1, 0], X_s[y_true == 1, 1], s=12, c="#D64550", label="Fraud", alpha=0.6)
    ax.set_xlabel("Amount (scaled)")
    ax.set_ylabel("Distance from Home (scaled)")
    ax.set_title(title)
    ax.legend()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_boundary(axes[0], svm_linear, X_test_s, y_test.values, f"Linear Kernel (Acc={accuracy_score(y_test, pred_linear):.3f})")
plot_boundary(axes[1], svm_rbf, X_test_s, y_test.values, f"RBF Kernel (Acc={accuracy_score(y_test, pred_rbf):.3f})")
plt.tight_layout()
plt.show()

**Interpretation:** The Linear kernel can only draw a straight boundary, so it misses fraud
transactions on the opposite side of the "legit" cluster. The RBF kernel wraps a **curved
boundary around the legit cluster**, correctly flagging fraud on all sides — this is the
kernel trick in action.

---
## Summary — Model Comparison

| Model | Task | Use Case | Key Metric |
|---|---|---|---|
| Linear Regression | Regression | Loan Amount Prediction | R² Score |
| Logistic Regression | Binary Classification | Loan Default Prediction | ROC-AUC |
| Decision Tree | Multi-class Classification | Investment Risk Category | Accuracy |
| SVM (RBF) | Non-linear Classification | Credit Card Fraud Detection | Accuracy / F1 |

Each model was chosen to match its strength: Linear Regression for continuous prediction,
Logistic Regression for a simple binary decision, Decision Tree for interpretable multi-class
splits, and SVM with the RBF kernel for a case where classes are not linearly separable.
